# Preprocessing Tutorial

This notebook runs the preprocessing workflow of the NUTMEG project.

Supported analysis modes:
- `RNA_ONLY`
- `MULTIOME`

Supported input sources:
- `10X_H5`
- `SEURAT_OBJECT`

Main workflow:
1. Initialize the `GRNPrep` object
2. Load the input data
3. Compute QC metrics
4. Filter cells and features
5. Run preprocessing and labeling
6. Export the unified NUTMEG input files

The final NUTMEG inputs are:
- `edge.csv`
- `node.csv`
- `metadata.csv`

A `labels.csv` file is also written during preprocessing as an auxiliary record of the cell labels.

In [8]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%%R
source("../src/nutmeg/preprocessing.R")

## Initialize workflow

Create a `GRNPrep` object to define the input data, analysis mode, output folder, and random seed used throughout the workflow.

### Arguments

- `mode`
  - `"RNA_ONLY"`: use for scRNA-seq datasets without ATAC data.
  - `"MULTIOME"`: use for paired RNA + ATAC single-cell datasets.

- `source`
  - `"10X_H5"`: load data from a 10x Genomics `.h5` file.
  - `"SEURAT_OBJECT"`: start from an existing Seurat object saved as `.rds`.

- `tenx_h5_path`: path to the 10x `.h5` input file. Required only when `source = "10X_H5"`.

- `tenx_frag_path`: path to the `fragments.tsv.gz` file. Required only when `mode = "MULTIOME"` and `source = "10X_H5"`.

- `rds_file`: path to an existing Seurat object saved as `.rds`. Required only when `source = "SEURAT_OBJECT"`.

- `experiment`: output directory for the current run.

- `seed`: random seed used for reproducibility.

In [10]:
%%R
pbmc <- GRNPrep$new(
  mode = "MULTIOME",
  source = "10X_H5",
  tenx_h5_path = "../data/raw/pbmc_unsorted_3k_filtered_feature_bc_matrix.h5",
  tenx_frag_path = "../data/raw/pbmc_unsorted_3k_atac_fragments.tsv.gz",
  experiment = "../results/test_multi",
  seed = 42
)



[GRNPrep] Initialized
 - experiment: ../results/test_multi
 - mode: MULTIOME
 - source: 10X_H5
 - seed: 42
 - csv_dir: /home/fotini/Desktop/lala/NUTMEG/results/test_multi/interm/csv
 - plot_dir: /home/fotini/Desktop/lala/NUTMEG/results/test_multi/interm/plots
 - rds_dir: /home/fotini/Desktop/lala/NUTMEG/results/test_multi/rds



## Load data and compute QC

This step loads the input data and computes quality-control metrics used later for filtering.

For RNA data, the workflow computes:
- `log_nCount_RNA`
- `percent.mt`

For `MULTIOME`, it also computes:
- `log_nCount_ATAC`
- `log_TSS.enrichment`
- `log_nucleosome_signal`

### `load()` options

- `keep_standard_chromosomes`
  - Default: `TRUE`
  - Keeps only standard human chromosomes when loading ATAC peaks in `MULTIOME` mode.

### `qc()` options

- `save_plots`: save QC histograms to disk. Default: `TRUE`.
- `filename`: output image filename. Default: `"qc_hist_before_filtering.png"`.
- `force`: recompute QC metrics even if they already exist. Default: `FALSE`.

### Output

After this step:
- The Seurat object is stored in `GRNPrep`.
- QC metrics are added to cell metadata.
- QC histograms are generated and optionally saved.

In [11]:
%%R
pbmc$
  load()$
  qc()

  |                                                  | 0 % ~calculating   |++                                                | 4 % ~32s           |++++                                              | 8 % ~27s           |++++++                                            | 12% ~38s           |++++++++                                          | 16% ~36s           |++++++++++                                        | 20% ~36s           |++++++++++++                                      | 24% ~34s           |++++++++++++++                                    | 28% ~33s           |++++++++++++++++                                  | 32% ~32s           |++++++++++++++++++                                | 36% ~29s           |++++++++++++++++++++                              | 40% ~28s           |++++++++++++++++++++++                            | 44% ~27s           |++++++++++++++++++++++++                          | 48% ~25s           |++++++++++++++++++++++++++                        | 52% ~22s 


[load] Loading data... (source=10X_H5, mode=MULTIOME)
Genome matrix has multiple modalities, returning a list of matrices for this genome
Computing hash
Checking for 3012 cell barcodes
[load] Done. Cells=3012

[qc] Computing QC metrics... (mode=MULTIOME)
Found 3012 cell barcodes
Done Processing 15 million lines
Extracting TSS positions
Extracting fragments at TSSs

Computing TSS enrichment score
[qc] Done. Plot saved -> ../results/test_multi/interm/plots/qc_hist_before_filtering.png
In addition: There were 24 warnings (use warnings() to see them)


## Filter cells and features

Filter cells using the QC metrics and remove low-detection features.  
In `RNA_ONLY` mode, the workflow filters cells and genes.  
In `MULTIOME` mode, ATAC-based cell filters and peak filtering are also applied.

#### `filter()` options

**RNA cell-level filters**

- `rna_log_ncount_min`  
  Lower bound for `log10(nCount_RNA)`.  
  Cells below this threshold are removed because they likely have very low RNA complexity.

- `rna_log_ncount_max`  
  Upper bound for `log10(nCount_RNA)`.  
  Cells above this threshold are removed as potential outliers or multiplets.

- `rna_mt_pct_max`  
  Maximum allowed mitochondrial percentage (`percent.mt`).  
  Cells above this threshold are removed because high mitochondrial signal often indicates low-quality or stressed cells.

**ATAC cell-level filters (`MULTIOME` only)**

- `atac_log_ncount_min`  
  Lower bound for `log10(nCount_ATAC)`.  
  Removes cells with very weak ATAC signal.

- `atac_log_ncount_max`  
  Upper bound for `log10(nCount_ATAC)`.  
  Removes extreme ATAC outliers.

- `atac_log_nucleo_max`  
  Maximum allowed `log10(nucleosome_signal)`.  
  High nucleosome signal can indicate poor-quality ATAC profiles.  

- `atac_log_tss_min`  
  Minimum allowed `log10(TSS.enrichment)`.  
  Cells below this threshold are removed because low TSS enrichment usually reflects poor chromatin accessibility quality.

**Feature-level filters**

- `min_cells_per_gene`  
  Keep only genes detected in at least this many cells.

- `min_cells_per_peak`  
  Keep only ATAC peaks detected in at least this many cells.  
  This option is relevant only in `MULTIOME` mode.

#### Output of this step

After filtering:
- low-quality cells are removed
- rarely detected genes are removed
- rarely detected peaks are removed in `MULTIOME` mode
- the filtered object is stored for preprocessing

Before choosing the appropriate values for a new dataset, inspect the QC histograms produced by `qc()`. The appropriate thresholds depend on sequencing depth, mitochondrial content, ATAC quality, and the biological system.

In [12]:
%%R
pbmc$filter(
  rna_log_ncount_min = 3,
  rna_log_ncount_max = 4,
  rna_mt_pct_max = 20,
  atac_log_ncount_min = 3,
  atac_log_ncount_max = 4.5,
  atac_log_nucleo_max = 0.3,
  atac_log_tss_min = 0.0,
  min_cells_per_gene = 3,
  min_cells_per_peak = 10
)


[filter] Filtering cells & features... (mode=MULTIOME)
[filter] Done. Cells: 3012 -> 2571
In addition: Warning messages:
1: Different cells and/or features from existing assay RNA 
2: Different cells and/or features from existing assay ATAC 


## Preprocess and label cells

This step normalizes RNA expression, reduces dimensionality, clusters cells, and assigns cell labels.

For `RNA_ONLY`, the workflow uses:
- SCT normalization
- PCA
- UMAP
- Leiden-style clustering

For `MULTIOME`, it uses:
- SCT normalization for RNA
- TF-IDF and LSI for ATAC
- Weighted nearest neighbors
- UMAP
- Clustering

### Labeling options

- `label_mode = "clusters"`
  - Uses the Seurat cluster identities as labels.
  - The labels are written to `cell_label`.

- `label_mode = "labels"`
  - Uses externally supplied cell labels from a CSV file.
  - The CSV must contain barcode and label columns.

The default label column is:

```r
labels_column_name = "cell_label"
```

This column is later read by `grn_exports()` and written into `metadata.csv`.

In [13]:
%%R
pbmc$preprocess(
  n_sct_features = 2000,
  n_pca_dims = 50,
  n_lsi_dims = 20,
  resolution = 0.5,
  label_mode = "labels",
  labels_file    = "../data/labels.csv",
  final_rds_filename = "pbmc_multi.rds"
)


[preprocess] Preprocessing + labeling (mode=MULTIOME, labels=labels)
[preprocess] Running SCTransform (n_features=2000) ...
[preprocess] Running PCA ...
Performing TF-IDF normalization
Running SVD
Scaling cell embeddings
[preprocess] Adding labels (mode=labels)
[preprocess] Subsetting: 2484/2484 labeled cells
[preprocess] labels.csv saved: /home/fotini/Desktop/lala/NUTMEG/results/test_multi/interm/csv/labels.csv
[preprocess] Labeled object saved: /home/fotini/Desktop/lala/NUTMEG/results/test_multi/rds/pbmc_multi.rds
In addition: There were 50 or more warnings (use warnings() to see the first 50)


## Export unified NUTMEG inputs

`grn_exports()` creates the three files required by NUTMEG:

| File | Contents |
|---|---|
| `edge.csv` | Candidate TF-to-CRE and CRE-to-gene relationships |
| `node.csv` | Per-cell observations of genes and CREs |
| `metadata.csv` | Cell metadata, labels, sample IDs, and optional coordinates |

The exporter uses:
- SCT variable features for genes
- ATAC variable features for peaks in `MULTIOME` mode
- SCT-normalized expression for RNA measurements
- TF-IDF-normalized accessibility for ATAC measurements


### Arguments

| Argument | Default | Meaning |
|---|---:|---|
| `motif_raw_threshold` | `5` | Minimum motif score for a TF-to-CRE edge |
| `jaspar_species` | `"Homo sapiens"` | JASPAR species |
| `jaspar_all_versions` | `FALSE` | Use only current JASPAR motif versions |
| `promoter_window_bp` | `2000` | Distance from TSS considered promoter |
| `edge_filename` | `"edge.csv"` | Output edge file |
| `node_filename` | `"node.csv"` | Output node file |
| `metadata_filename` | `"metadata.csv"` | Output metadata file |
| `labels_column_name` | `"cell_label"` | Metadata column containing cell labels |
| `sample_id` | `NULL` | Fixed sample ID for all cells |
| `sample_column` | `NULL` | Metadata column containing per-cell sample IDs |
| `coordinate_columns` | All `NA` | Optional mapping of spatial coordinates |


For an object containing multiple samples, use:

```r
sample_column = "orig.ident"
```

Do not map UMAP coordinates to `x_coord` and `y_coord` unless you intentionally want to treat them as spatial coordinates.

In [14]:
%%R
pbmc$grn_exports(
  labels_column_name = "cell_label",
  sample_id = "pbmc_multi"
)

  |                                                  | 0 % ~calculating   |+                                                 | 1 % ~38m 46s       |+                                                 | 2 % ~30m 45s       |++                                                | 3 % ~27m 02s       |++                                                | 4 % ~32m 19s       |+++                                               | 5 % ~33m 24s       |+++                                               | 6 % ~34m 40s       |++++                                              | 7 % ~36m 59s       |++++                                              | 8 % ~35m 22s       |+++++                                             | 9 % ~34m 12s       |+++++                                             | 10% ~33m 53s       |++++++                                            | 11% ~35m 47s       |++++++                                            | 12% ~36m 02s       |+++++++                                           | 13% ~35m 


[grn_exports] Building unified NUTMEG inputs... (mode=MULTIOME)
[grn_exports] Gene-side set = 2000 genes.
[grn_exports] Peak-side set = 17970 peaks.
[grn_exports] Running LinkPeaks (ATAC -> SCT) ...
Found gene coordinates for 1699 genes
Testing 1999 genes and 44943 peaks
[grn_exports] MULTIOME CRE->Gene edges: 1863 | unique CREs=1515 | unique genes=878
[grn_exports] Motif matching on CREs (JASPAR2020) ... this can take a while.
[grn_exports] TF->CRE edges after filtering: 24875 (raw_score > 5)
[grn_exports] Creating unified NUTMEG inputs...
[grn_exports] Unified inputs complete:
 - Candidate edges: 26738
 - Node observations: 1088854
 - Cells in metadata: 2484
 - Edge file: /home/fotini/Desktop/lala/NUTMEG/results/test_multi/interm/csv/edge.csv
 - Node file: /home/fotini/Desktop/lala/NUTMEG/results/test_multi/interm/csv/node.csv
 - Metadata file: /home/fotini/Desktop/lala/NUTMEG/results/test_multi/interm/csv/metadata.csv
